# NHL Analytics Hub — Data Collection & Loading
This notebook fetches data from the NHL public API, parses it, and loads it into a PostgreSQL database across 7 tables: teams, standings, players, games, game_stats, skater_season_stats, goalie_season_stats.

In [33]:
import requests

response = requests.get("https://api-web.nhle.com/v1/standings/now")
data = response.json()
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['wildCardIndicator', 'standingsDateTimeUtc', 'standings'])


# Phase 1: Fetch and inspect standings data
Fetching from standings now, which returns both team identity info (used for the `teams` table) and current season records (used for `standings`).

In [34]:
teams_list = data["standings"]
print(len(teams_list))
print(teams_list[0])

32
{'clinchIndicator': 'p', 'conferenceAbbrev': 'W', 'conferenceHomeSequence': 1, 'conferenceL10Sequence': 2, 'conferenceName': 'Western', 'conferenceRoadSequence': 1, 'conferenceSequence': 1, 'date': '2026-04-17', 'divisionAbbrev': 'C', 'divisionHomeSequence': 1, 'divisionL10Sequence': 1, 'divisionName': 'Central', 'divisionRoadSequence': 1, 'divisionSequence': 1, 'gameTypeId': 2, 'gamesPlayed': 82, 'goalDifferential': 99, 'goalDifferentialPctg': 1.207317, 'goalAgainst': 203, 'goalFor': 302, 'goalsForPctg': 3.682927, 'homeGamesPlayed': 41, 'homeGoalDifferential': 49, 'homeGoalsAgainst': 108, 'homeGoalsFor': 157, 'homeLosses': 9, 'homeOtLosses': 6, 'homePoints': 58, 'homeRegulationPlusOtWins': 25, 'homeRegulationWins': 25, 'homeTies': 0, 'homeWins': 26, 'l10GamesPlayed': 10, 'l10GoalDifferential': 14, 'l10GoalsAgainst': 20, 'l10GoalsFor': 34, 'l10Losses': 2, 'l10OtLosses': 1, 'l10Points': 15, 'l10RegulationPlusOtWins': 6, 'l10RegulationWins': 6, 'l10Ties': 0, 'l10Wins': 7, 'leagueHomeS

# Phase 2: Parse team data
Extracting team identity fields (abbreviation, name, conference, division, logo) into a clean list of dictionaries matching the `teams` table schema.

In [35]:
teams_data = []
for team in data ["standings"]:
    teams_data.append(dict(
        division_name=team["divisionName"],
        conference_name=team["conferenceName"],     
        team_name=team["teamName"]["default"],
        team_abbrev=team["teamAbbrev"]["default"],
        logo_url=team["teamLogo"]
    ))
teams_data


[{'division_name': 'Central',
  'conference_name': 'Western',
  'team_name': 'Colorado Avalanche',
  'team_abbrev': 'COL',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/COL_light.svg'},
 {'division_name': 'Metropolitan',
  'conference_name': 'Eastern',
  'team_name': 'Carolina Hurricanes',
  'team_abbrev': 'CAR',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/CAR_light.svg'},
 {'division_name': 'Central',
  'conference_name': 'Western',
  'team_name': 'Dallas Stars',
  'team_abbrev': 'DAL',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/DAL_light.svg'},
 {'division_name': 'Atlantic',
  'conference_name': 'Eastern',
  'team_name': 'Buffalo Sabres',
  'team_abbrev': 'BUF',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/BUF_light.svg'},
 {'division_name': 'Atlantic',
  'conference_name': 'Eastern',
  'team_name': 'Tampa Bay Lightning',
  'team_abbrev': 'TBL',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/TBL_light.svg'},
 {'division_name': 'Atlantic',
  'co

# Phase 3: Connect to PostgreSQL and create schema
Connecting to the `nhl_analytics` database and creating all 7 tables (if they don't already exist).

In [36]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="nhl_analytics",
    user="kamila",
    password="1234"
)
cursor = conn.cursor()

# Create all 7 tables
Creating the full schema — teams, standings, players, games, game_stats, skater_season_stats, and goalie_season_stats — using `IF NOT EXISTS` so this cell is safe to re-run without erroring if the tables already exist.

In [37]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS teams (
    team_id SERIAL PRIMARY KEY,
    team_abbrev VARCHAR(10) UNIQUE,
    team_name VARCHAR(100),
    conference_name VARCHAR(50),
    division_name VARCHAR(50),
    logo_url TEXT
);
""")
conn.commit()
print("team table Ready")

team table Ready


In [38]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS standings (
    standing_id SERIAL PRIMARY KEY,
    team_id INT REFERENCES TEAMS(team_id),
    season varchar(20),
    games_played INT,
    wins INT,
    losses INT,
    ot_losses INT,
    points INT,
    goals_for INT,
    goals_against INT,
    home_wins INT,
    away_wins INT,
    streak_type VARCHAR(20),
    streak_count INT
);
""")
conn.commit()
print("standings table Ready")

standings table Ready


In [39]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS players (
    player_id BIGINT PRIMARY KEY,
    team_id INT REFERENCES TEAMS(team_id),
    first_name VARCHAR(100),
    last_name VARCHAR(100),
    "position" VARCHAR(10),
    jersey_number INT,
    birth_date DATE,
    birth_country VARCHAR(10),
    height_cm REAL,
    weight_kg REAL,
    shoots_catches VARCHAR(5),
    headshot_url TEXT
);
""")
conn.commit()
print("players table Ready")

players table Ready


In [40]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS games (
     game_id BIGINT PRIMARY KEY,
    season VARCHAR(20),
    game_type INT,
    game_date DATE,
    home_team_id INT REFERENCES TEAMS(team_id),
    away_team_id INT REFERENCES TEAMS(team_id),
    home_score INT,
    away_score INT,
    game_state VARCHAR(20),
    venue_name VARCHAR(150)
);
""")
conn.commit()
print("games table Ready")

games table Ready


In [41]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS game_stats(
    stat_id serial PRIMARY KEY,
    game_id BIGINT REFERENCES GAMES(game_id),
    player_id BIGINT REFERENCES Players(player_id),
    team_id INT REFERENCES TEAMS(team_id),
    goals INT,
    assists INT,
    points INT,
    shots_on_goal INT,
    penalty_min INT,
    toi VARCHAR(10),
    plus_minus INT
);
""")
conn.commit()
print("game_stats table Ready")

game_stats table Ready


In [42]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS skater_season_stats (
 stat_id serial PRIMARY KEY,
    player_id BIGINT REFERENCES Players(player_id),
    season VARCHAR(20),
    team_id INT REFERENCES TEAMS(team_id),
    games_played INT,
    goals INT,
    assists INT,
    points INT,
    plus_minus INT,
    penalty_min INT,
    shots INT,
    avg_toi VARCHAR(10)
);
""")
conn.commit()
print("skater_season_stats table Ready")

skater_season_stats table Ready


In [43]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS goalie_season_stats (
    stat_id serial PRIMARY KEY,
    player_id BIGINT REFERENCES Players(player_id),
    season VARCHAR(20),
    team_id INT REFERENCES TEAMS(team_id),
    games_played INT,
    wins INT,
    losses INT,
    ot_losses INT,
    save_pct FLOAT,          
    goals_against_avg FLOAT,
    shutouts INT,       
    saves INT
);
""")
conn.commit()
print("goalie_season_stats table Ready")

goalie_season_stats table Ready


# Load teams into the database

In [44]:
for team in teams_data:
    cursor.execute(
        """
        INSERT INTO teams (team_abbrev, team_name, conference_name, division_name, logo_url)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (team_abbrev) DO NOTHING
        """,
        (team['team_abbrev'], team['team_name'], team['conference_name'], team['division_name'], team['logo_url'])
    )

conn.commit()
print("Teams inserted")

Teams inserted


## Build a team_abbrev - team_id lookup
The NHL API only provides team abbreviations, not our database's auto-generated numeric IDs. This lookup translates between the two, needed for every table with a team_id foreign key.

In [45]:
cursor.execute("SELECT team_id, team_abbrev FROM teams")
rows = cursor.fetchall()
team_id_map = {abbrev: team_id for team_id, abbrev in rows}
print(team_id_map)

{'COL': 1, 'CAR': 2, 'DAL': 3, 'BUF': 4, 'TBL': 5, 'MTL': 6, 'MIN': 7, 'BOS': 8, 'OTT': 9, 'PIT': 10, 'PHI': 11, 'WSH': 12, 'VGK': 13, 'EDM': 14, 'UTA': 15, 'DET': 16, 'CBJ': 17, 'ANA': 18, 'NYI': 19, 'LAK': 20, 'NJD': 21, 'STL': 22, 'NSH': 23, 'SJS': 24, 'FLA': 25, 'WPG': 26, 'SEA': 27, 'TOR': 28, 'CGY': 29, 'NYR': 30, 'CHI': 31, 'VAN': 32}


# Parse and load standings data
Using the same /standings/now response, extracting win/loss/points data this time, with team_id resolved via the lookup dictionary.

In [46]:
standings_data = []
for team in data["standings"]:
    abbrev = team["teamAbbrev"]["default"]
    standings_data.append(dict(
        team_id=team_id_map[abbrev],
        season = str(team["seasonId"]),
        games_played = team["gamesPlayed"],
        wins = team["wins"],
        losses = team["losses"],
        ot_losses = team["otLosses"],
        points = team["points"],
        goals_for = team["goalFor"],
        goals_against = team["goalAgainst"],
        home_wins = team["homeWins"],
        away_wins = team["roadWins"],
        streak_type = team["streakCode"],
        streak_count = team["streakCount"]
    ))
print(len(standings_data))
print(standings_data[0])


32
{'team_id': 1, 'season': '20252026', 'games_played': 82, 'wins': 55, 'losses': 16, 'ot_losses': 11, 'points': 121, 'goals_for': 302, 'goals_against': 203, 'home_wins': 26, 'away_wins': 29, 'streak_type': 'W', 'streak_count': 3}


In [58]:
for s in standings_data:
    cursor.execute(
        """
        INSERT INTO standings (team_id, season, games_played, wins, losses, ot_losses,
                                points, goals_for, goals_against, home_wins, away_wins,
                                streak_type, streak_count)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """,
        (s['team_id'], s['season'], s['games_played'], s['wins'], s['losses'], s['ot_losses'],
         s['points'], s['goals_for'], s['goals_against'], s['home_wins'], s['away_wins'],
         s['streak_type'], s['streak_count'])
    )

conn.commit()
print("Standings inserted")

Standings inserted


In [48]:
players_data = []

for abbrev in team_id_map.keys():
    roster = requests.get(f"https://api-web.nhle.com/v1/roster/{abbrev}/current").json()
    for group in ['forwards', 'defensemen', 'goalies']:
        for p in roster[group]:
            players_data.append(dict(
                player_id = p['id'],
                team_id = team_id_map[abbrev],
                first_name = p['firstName']['default'],
                last_name = p['lastName']['default'],
                position = p['positionCode'],
                jersey_number = p.get('sweaterNumber'),
                birth_date = p.get('birthDate'),
                birth_country = p.get('birthCountry'),
                height_cm = p.get('heightInCentimeters'),
                weight_kg = p.get('weightInKilograms'),
                shoots_catches = p.get('shootsCatches'),
                headshot_url = p.get('headshot')
            ))

print(len(players_data))
print(players_data[0])

807
{'player_id': 8482947, 'team_id': 1, 'first_name': 'Zakhar', 'last_name': 'Bardakov', 'position': 'C', 'jersey_number': 93, 'birth_date': '2001-02-24', 'birth_country': 'RUS', 'height_cm': 188, 'weight_kg': 90, 'shoots_catches': 'L', 'headshot_url': 'https://assets.nhle.com/mugs/nhl/20262027/COL/8482947.png'}


# Phase 1-3: Players
Fetching each team's roster individually from /v1/roster/{team_abbrev}/current. Each roster response groups players into forwards, defensemen, and goalies — flattened into a single list here. Missing fields (e.g. jersey number) are handled safely with .get().

In [49]:
for p in players_data:
    cursor.execute(
        """
        INSERT INTO players (player_id, team_id, first_name, last_name, "position",
                              jersey_number, birth_date, birth_country, height_cm,
                              weight_kg, shoots_catches, headshot_url)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (player_id) DO NOTHING
        """,
        (p['player_id'], p['team_id'], p['first_name'], p['last_name'], p['position'],
         p['jersey_number'], p['birth_date'], p['birth_country'], p['height_cm'],
         p['weight_kg'], p['shoots_catches'], p['headshot_url'])
    )

conn.commit()
print("Players inserted")

Players inserted


# Phase 1-3: Games
Fetching each team's full season schedule from /v1/club-schedule-season/{team_abbrev}/20252026. Since each game appears in both teams' schedules, a dictionary keyed by game_id is used to avoid inserting duplicates.

In [50]:
games_dict = {}

for abbrev in team_id_map.keys():
    schedule = requests.get(f"https://api-web.nhle.com/v1/club-schedule-season/{abbrev}/20252026").json()
    for g in schedule['games']:
        game_id = g['id']
        games_dict[game_id] = dict(
            game_id = game_id,
            season = str(g['season']),
            game_type = g['gameType'],
            game_date = g['gameDate'],
            home_team_id = team_id_map[g['homeTeam']['abbrev']],
            away_team_id = team_id_map[g['awayTeam']['abbrev']],
            home_score = g['homeTeam'].get('score'),
            away_score = g['awayTeam'].get('score'),
            game_state = g['gameState'],
            venue_name = g['venue']['default']
        )

games_data = list(games_dict.values())
print(len(games_data))
print(games_data[0])

1498
{'game_id': 2025010103, 'season': '20252026', 'game_type': 1, 'game_date': '2025-09-21', 'home_team_id': 15, 'away_team_id': 1, 'home_score': 1, 'away_score': 5, 'game_state': 'FINAL', 'venue_name': 'Magness Arena'}


In [51]:
for g in games_data:
    cursor.execute(
        """
        INSERT INTO games (game_id, season, game_type, game_date, home_team_id, away_team_id,
                            home_score, away_score, game_state, venue_name)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (game_id) DO NOTHING
        """,
        (g['game_id'], g['season'], g['game_type'], g['game_date'], g['home_team_id'], g['away_team_id'],
         g['home_score'], g['away_score'], g['game_state'], g['venue_name'])
    )

conn.commit()
print("Games inserted")

Games inserted


# Phase 1-3: Game stats (from provided dataset)
Per-game skater statistics were provided as a pre-fetched JSON file rather than fetched live, due to the large number of API calls required (~1,300+ games). Rows referencing players no longer on any current roster are skipped, since they violate the foreign key constraint on players.

In [52]:
import json

with open('../data/game_stats.json') as f:
    game_stats_data = json.load(f)

print(len(game_stats_data))

47408


In [59]:
conn.rollback()

cursor.execute("SELECT player_id FROM players")
valid_player_ids = set(row[0] for row in cursor.fetchall())
print(len(valid_player_ids))

inserted = 0
skipped = 0

for s in game_stats_data:
    if s['player_id'] not in valid_player_ids:
        skipped += 1
        continue
    cursor.execute(
        """
        INSERT INTO game_stats (game_id, player_id, team_id, goals, assists, points,
                                 shots_on_goal, penalty_min, toi, plus_minus)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """,
        (s['game_id'], s['player_id'], s['team_id'], s['goals'], s['assists'], s['points'],
         s['shots_on_goal'], s['penalty_min'], s['toi'], s['plus_minus'])
    )
    inserted += 1

conn.commit()
print(f"Inserted: {inserted}, Skipped (player not found): {skipped}")

807
Inserted: 47387, Skipped (player not found): 21


# Phase 1-3: Skater season stats (from provided dataset)
Full-season skater totals were provided as a pre-fetched JSON file, same approach as game_stats. Rows for players not in the current roster are skipped.

In [54]:
with open('../data/skater_season_stats.json') as f:
    skater_season_stats_data = json.load(f)

print(len(skater_season_stats_data))
print(skater_season_stats_data[0])

714
{'stat_id': 1, 'player_id': 8470613, 'season': '20252026', 'team_id': 1, 'games_played': 82, 'goals': 12, 'assists': 23, 'points': 35, 'plus_minus': 33, 'penalty_min': 32, 'shots': 172, 'avg_toi': None}


In [67]:
inserted = 0
skipped = 0

for s in skater_season_stats_data:
    if s['player_id'] not in valid_player_ids:
        skipped += 1
        continue
    cursor.execute(
        """
        INSERT INTO skater_season_stats (player_id, season, team_id, games_played, goals,
                                          assists, points, plus_minus, penalty_min, shots, avg_toi)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """,
        (s['player_id'], s['season'], s['team_id'], s['games_played'], s['goals'],
         s['assists'], s['points'], s['plus_minus'], s['penalty_min'], s['shots'], s['avg_toi'])
    )
    inserted += 1

conn.commit()
print(f"Inserted: {inserted}, Skipped: {skipped}")

Inserted: 713, Skipped: 1


# Phase 1-3: Goalie season stats (from provided dataset)
Full-season goalie totals were provided as a pre-fetched JSON file, same approach as skater_season_stats.

In [56]:
with open('../data/goalie_season_stats.json') as f:
    goalie_season_stats_data = json.load(f)

print(len(goalie_season_stats_data))
print(goalie_season_stats_data[0])

81
{'stat_id': 1, 'player_id': 8475809, 'season': '20252026', 'team_id': 1, 'games_played': 45, 'wins': 31, 'losses': 6, 'ot_losses': 6, 'save_pct': 0.921317, 'goals_against_avg': None, 'shutouts': 4, 'saves': None}


In [68]:
inserted = 0
skipped = 0

for g in goalie_season_stats_data:
    if g['player_id'] not in valid_player_ids:
        skipped += 1
        continue
    cursor.execute(
        """
        INSERT INTO goalie_season_stats (player_id, season, team_id, games_played, wins, losses,
                                          ot_losses, save_pct, goals_against_avg, shutouts, saves)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """,
        (g['player_id'], g['season'], g['team_id'], g['games_played'], g['wins'], g['losses'],
         g['ot_losses'], g['save_pct'], g['goals_against_avg'], g['shutouts'], g['saves'])
    )
    inserted += 1

conn.commit()
print(f"Inserted: {inserted}, Skipped: {skipped}")

Inserted: 81, Skipped: 0
